In [ ]:
import gc
import sklearn
import numpy as np
import keras_tuner
import tensorflow as tf
import matplotlib.pyplot as plt

import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("Modules"))))

import Modules.constants as constants 
import Modules.ds_loader as ds_loader

train_loader, val_loader, test_loader = ds_loader.load_tf_data()


2025-04-15 16:39:51.987974: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-15 16:39:52.081671: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744727992.134292  117926 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744727992.146565  117926 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744727992.242850  117926 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
"""# 1-D convolutional ResNet model 
# https://pmc.ncbi.nlm.nih.gov/articles/PMC10128986/#sec012
class Resnet(keras_tuner.HyperModel):
    def residual_block(self, inputs, c_units, p_units, k_units):
        # C1 BLOCK
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(inputs)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        # SC
        s = tf.keras.layers.Conv1D(filters=c_units, kernel_size=1, strides=1, padding='same')(inputs)
        x = tf.keras.layers.Add()([x, s])
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(p_units, strides=2)(x)
        return x


    def build(self, hp):
        gc.collect()
        tf.keras.backend.clear_session()
        # HYPERPARAMS
        n_layer = 3
        k_units = 3
        p_units = 5

        c_units = hp.Choice("c_units", [64])
        d_units_0 = hp.Choice("d_units_0", [1024])
        d_units_1 = hp.Choice('d_units_coef', [2,4,8])
        dropout_0 = hp.Float('dropout_0', min_value = 0.3, max_value=0.5, step=0.05)
        dropout_1 = hp.Float('dropout_1', min_value = 0.3, max_value=0.5, step=0.05)
        
        # INPUT LAYER
        inputs = tf.keras.Input(shape=(500,12))
        
        # RESIDUALS
        x = self.residual_block(inputs, c_units, p_units, k_units)
        filter_size = c_units
        for i in range(1, n_layer):
            filter_size *= 2  
            x = self.residual_block(x, filter_size, p_units, k_units)

        # CLASSIFIER
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(d_units_0, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_0)(x)  
        x = tf.keras.layers.Dense(d_units_0 // d_units_1, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_1)(x)  

        # OUTPUT
        outputs = tf.keras.layers.Dense(4, activation='softmax')(x)
        
        model = tf.keras.Model(inputs, outputs)
        optimizer = tf.keras.optimizers.Adam(
                        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-3),
                        weight_decay=hp.Choice('weight_decay',[1e-3,1e-4,1e-5,0.0])
                    )
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        
        
        return model
    
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            batch_size= hp.Choice("batch_size", [32]),
            *args,
            **kwargs,
        ) """

'# 1-D convolutional ResNet model \n# https://pmc.ncbi.nlm.nih.gov/articles/PMC10128986/#sec012\nclass Resnet(keras_tuner.HyperModel):\n    def residual_block(self, inputs, c_units, p_units, k_units):\n        # C1 BLOCK\n        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding=\'same\')(inputs)\n        x = tf.keras.layers.ReLU()(x)\n        x = tf.keras.layers.BatchNormalization()(x)\n\n        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding=\'same\')(x)\n        x = tf.keras.layers.ReLU()(x)\n        x = tf.keras.layers.BatchNormalization()(x)\n        # SC\n        s = tf.keras.layers.Conv1D(filters=c_units, kernel_size=1, strides=1, padding=\'same\')(inputs)\n        x = tf.keras.layers.Add()([x, s])\n        x = tf.keras.layers.ReLU()(x)\n        x = tf.keras.layers.BatchNormalization()(x)\n        x = tf.keras.layers.MaxPooling1D(p_units, strides=2)(x)\n        return x\n\n\n    def build(self, hp):\n        

In [3]:
class Resnet(keras_tuner.HyperModel):
    def residual_block(self, inputs, c_units, p_units, k_units):
        # C1 BLOCK
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=6, strides=1, padding='same')(inputs)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(6, 6, padding="same")(x)
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(p_units, p_units, padding="same")(x)
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(p_units, p_units, padding="same")(x)
    
        return x


    def build(self, hp):
        gc.collect()
        tf.keras.backend.clear_session()
        # HYPERPARAMS
        n_layer = 1
        k_units = 3
        p_units = 3

        c_units = hp.Choice("c_units", [32,64,128,256])
        d_units_0 = hp.Choice("d_units_0", [64,128,256,512,1024])
        d_units_1 = hp.Choice('d_units_coef', [2,4])
        dropout_0 = hp.Float('dropout_0', min_value = 0.2, max_value=0.5, step=0.05)
        dropout_1 = hp.Float('dropout_1', min_value = 0.2, max_value=0.5, step=0.05)
        
        # INPUT LAYER
        inputs = tf.keras.Input(shape=(constants.FINAL_SIZE,12))
        
        # RESIDUALS
        x = self.residual_block(inputs, c_units, p_units, k_units)
        filter_size = c_units
        for i in range(1, n_layer):
            #filter_size *= 2  
            x = self.residual_block(x, filter_size, p_units, k_units)

        # CLASSIFIER
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(d_units_0, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_0)(x)  
        x = tf.keras.layers.Dense(d_units_0 // d_units_1, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_1)(x)  

        # OUTPUT
        outputs = tf.keras.layers.Dense(4, activation='softmax')(x)
        
        model = tf.keras.Model(inputs, outputs)
        optimizer = tf.keras.optimizers.Adam(
                        learning_rate=hp.Float('learning_rate', min_value=1e-5, max_value=1e-3),
                        weight_decay=hp.Choice('weight_decay',[1e-3,1e-4,1e-5,0.0])
                    )
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        
        
        return model
    
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            batch_size= hp.Choice("batch_size", [32]),
            *args,
            **kwargs,
        ) 

In [ ]:
RDIR="../src/Results/RES_500_01/" 
MDIR= RDIR + "RES_W500_01.keras"
CDIR= RDIR + "C_RES_W500_01.keras"
CVDIR = RDIR + "RES_W500_01_CV.keras"

tuner = keras_tuner.Hyperband(
    Resnet(),
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    overwrite=False,
    directory=RDIR,
    project_name="RES_W500_01",
)
tuner.search_space_summary()


[ !! ]Skipping MUSE_20180113_124215_52000.csv: Unexpected shape (1926, 12)
[ !! ]Skipping MUSE_20180114_124230_39000.csv: Contains NaNs
[ !! ]Skipping MUSE_20180210_130454_71000.csv: Contains NaNs
[ !! ]Skipping MUSE_20180114_080214_06000.csv: Contains NaNs
[ OK ] Loaded 7404 samples with shape (5000, 12)
[ !! ]Skipping MUSE_20180712_152022_92000.csv: Contains NaNs
[ !! ]Skipping MUSE_20180113_180425_75000.csv: Contains NaNs
[ !! ]Skipping MUSE_20180712_151357_86000.csv: Contains NaNs
[ OK ] Loaded 1596 samples with shape (5000, 12)
[ !! ]Skipping MUSE_20180712_151353_58000.csv: Contains NaNs
[ !! ]Skipping MUSE_20180120_121805_89000.csv: Contains NaNs
[ OK ] Loaded 1595 samples with shape (5000, 12)
Unique classes in y: [0 1 2 3]
Datatype: float32 int32
Min and Max of X_train: -16721.0, 20584.0
Min and Max of X_val: -10988.0, 13647.0
Min and Max of X_test: -15385.0, 15463.0
NaNs in X: 0
Infs in X: 0
Class distribution before SMOTE: Counter({np.int32(2): 2720, np.int32(1): 1582, np.int

I0000 00:00:1744728067.975869  117926 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2261 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Reloading Tuner from ../src/Results/RES_500_01/RES_W500_01/tuner0.json
Search space summary
Default search space size: 8
c_units (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64, 128, 256], 'ordered': True}
d_units_0 (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128, 256, 512, 1024], 'ordered': True}
d_units_coef (Choice)
{'default': 2, 'conditions': [], 'values': [2, 4], 'ordered': True}
dropout_0 (Float)
{'default': 0.2, 'conditions': [], 'min_value': 0.2, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
dropout_1 (Float)
{'default': 0.2, 'conditions': [], 'min_value': 0.2, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
learning_rate (Float)
{'default': 1e-05, 'conditions': [], 'min_value': 1e-05, 'max_value': 0.001, 'step': None, 'sampling': 'linear'}
weight_decay (Choice)
{'default': 0.001, 'conditions': [], 'values': [0.001, 0.0001, 1e-05, 0.0], 'ordered': True}
batch_size (Choice)
{'default': 32, 'conditions': [], 'values': [32], 'ordered': Tru

Traceback (most recent call last):
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/tuners/hyperband.py", line 427, in run_trial
    return super().run_trial(trial, *fit_args, **fit_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/tuner.py", line 314, in run_trial
    obj_value = self._build_and_fit_model(trial, *args, **copied_kwargs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/tuners/hyperband.py", line 427, in run_trial
    return super().run_trial(trial, *fit_args, **fit_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/tuner.py", line 314, in run_trial
    obj_value = self._build_and_fit_model(trial, *args, **copi

In [ ]:
callback_list = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy",mode="max", restore_best_weights=True,patience=5, verbose=0),
    tf.keras.callbacks.ModelCheckpoint(filepath=CDIR,monitor='val_accuracy', save_best_only=True, save_weights_only=False,    
    verbose=0)
]

tuner.search(
    train_loader, 
    epochs = 150,
    validation_data=(val_loader),
    callbacks=callback_list 
)

In [ ]:
tuner.results_summary()

In [ ]:
models = tuner.get_best_models(num_models=1)
best_model = models[0]
best_model.summary()
best_model.save(MDIR) 

In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 
print(best_hps.values)

In [ ]:
test_loss, test_accuracy = best_model.evaluate(test_loader, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
X_test_list, y_test_list = [], []
X_train_list, y_train_list= [], []

for batch_x, batch_y in test_loader:
    X_test_list.append(batch_x.numpy())
    y_test_list.append(batch_y.numpy())
for batch_x, batch_y in train_loader:
    X_train_list.append(batch_x.numpy())
    y_train_list.append(batch_y.numpy())

X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

In [ ]:
y_pred = best_model.predict(X_test)

if y_pred.shape[1] == 1:  
    y_pred_binary = (y_pred > 0.5).astype(int)
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred)  
else:
    y_pred_binary = np.argmax(y_pred, axis=1)  
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred, multi_class='ovr')

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred_binary))
print(f"AUC: {auc}")

y_train_pred = best_model.predict(X_train)
y_train_pred = np.argmax(y_train_pred, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns
y_pred_class = np.argmax(y_pred, axis=1)  
cm = sklearn.metrics.confusion_matrix(y_test, y_pred_class, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
kfold = sklearn.model_selection.KFold(n_splits=10, shuffle=True, random_state=42)
fold_accuracies = []
fold_histories = []

best_accuracy = 0.0
best_model = None  

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1} ---")

    fold_callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)
    ]
    X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_tr, y_val_fold = y_train[train_idx], y_train[val_idx]

    model = Resnet().build(best_hps)

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val_fold, y_val_fold),
        epochs=100,
        callbacks=fold_callbacks,
        verbose=1
    )

    val_loss, val_accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    print(f"Fold {fold+1} Validation Accuracy: {val_accuracy:.4f}")
    fold_accuracies.append(val_accuracy)
    fold_histories.append(history)

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = model
        model.save(CVDIR) 
        print(f"Saved best model from Fold {fold+1} with Accuracy: {val_accuracy:.4f}")


In [ ]:
print("Cross-validation accuracies:", fold_accuracies)
print("Average CV accuracy:", np.mean(fold_accuracies))
print("Max CV accuracy:", np.max(fold_accuracies))

In [ ]:
cv_model = tf.keras.models.load_model(CVDIR)
cv_model.evaluate(X_test, y_test)

In [ ]:
test_loss, test_accuracy = cv_model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred = cv_model.predict(X_test)

if y_pred.shape[1] == 1:  
    y_pred_binary = (y_pred > 0.5).astype(int)
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred)  
else:
    y_pred_binary = np.argmax(y_pred, axis=1)  
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred, multi_class='ovr')

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred_binary))
print(f"AUC: {auc}")

y_train_pred = cv_model.predict(X_train)
y_train_pred = np.argmax(y_train_pred, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
y_pred_probs = cv_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred))

auc = sklearn.metrics.roc_auc_score(y_test, y_pred_probs, multi_class='ovr')
print(f"AUC (Test): {auc:.4f}")

y_train_probs = cv_model.predict(X_train)
y_train_pred = np.argmax(y_train_probs, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns

cm = sklearn.metrics.confusion_matrix(y_test, y_pred, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ax[0].plot(history.history['accuracy'], label='accuracy')
ax[0].plot(history.history['val_accuracy'], label='val_accuracy')
ax[0].set_title('Accuracy vs Val Accuracy')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Accuracy')
ax[0].legend()

ax[1].plot(history.history['loss'], label='loss')
ax[1].plot(history.history['val_loss'], label='val_loss')
ax[1].set_title('Loss vs Val Loss')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()